# EMIPredict AI: Exploratory Data Analysis
400,000 Financial Records Analysis

In [ ]:
# %% [markdown]
# # EMIPredict AI: Exploratory Data Analysis
# 400,000 Financial Records Analysis

# %% 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('../data/emi_dataset.csv')
print(f"Dataset Shape: {df.shape}")
print(f"Memory Usage: {df.memory_usage().sum() / 1024**2:.2f} MB")

# %% [markdown]
# ## 1. Data Overview

# %%
df.head()

# %%
df.info()

# %%
df.describe()

# %% [markdown]
# ## 2. Target Variable Analysis

# %%
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Classification target
df['emi_eligibility'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'orange', 'red'])
axes[0].set_title('EMI Eligibility Distribution')
axes[0].set_xlabel('Eligibility Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Add percentage labels
for i, v in enumerate(df['emi_eligibility'].value_counts().values):
    axes[0].text(i, v + 500, f'{v/len(df)*100:.1f}%', ha='center', fontsize=10)

# Regression target
axes[1].hist(df['max_monthly_emi'], bins=50, color='blue', alpha=0.7, edgecolor='black')
axes[1].axvline(df['max_monthly_emi'].mean(), color='red', linestyle='--', label=f'Mean: {df["max_monthly_emi"].mean():.0f}')
axes[1].set_title('Max Monthly EMI Distribution')
axes[1].set_xlabel('Max Monthly EMI (INR)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/eda_target_distribution.png', dpi=150)
plt.show()

# %% [markdown]
# ## 3. EMI Scenario Analysis

# %%
scenario_stats = df.groupby('emi_scenario').agg({
    'requested_amount': ['mean', 'median', 'std'],
    'requested_tenure': ['mean', 'median', 'std'],
    'emi_eligibility': lambda x: (x == 'Eligible').mean() * 100
}).round(2)

scenario_stats.columns = ['Amount_Mean', 'Amount_Median', 'Amount_Std', 
                         'Tenure_Mean', 'Tenure_Median', 'Tenure_Std',
                         'Eligibility_Rate_%']
print(scenario_stats)

# %%
# Visualization of scenario statistics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount by scenario
df.boxplot(column='requested_amount', by='emi_scenario', ax=axes[0])
axes[0].set_title('Requested Amount by EMI Scenario')
axes[0].set_ylabel('Amount (INR)')
axes[0].set_xlabel('EMI Scenario')
axes[0].tick_params(axis='x', rotation=45)

# Eligibility rate by scenario
eligibility_by_scenario = df.groupby('emi_scenario')['emi_eligibility'].apply(
    lambda x: (x == 'Eligible').mean() * 100
).sort_values(ascending=True)

eligibility_by_scenario.plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('Eligibility Rate by EMI Scenario')
axes[1].set_xlabel('Eligibility Rate (%)')
axes[1].set_ylabel('EMI Scenario')

plt.tight_layout()
plt.savefig('../reports/eda_scenario_analysis.png', dpi=150)
plt.show()

# %% [markdown]
# ## 4. Credit Score Analysis

# %%
# Credit score distribution by eligibility
fig, ax = plt.subplots(figsize=(10, 6))

for status in df['emi_eligibility'].unique():
    subset = df[df['emi_eligibility'] == status]['credit_score']
    ax.hist(subset, bins=30, alpha=0.5, label=status, density=True)

ax.set_title('Credit Score Distribution by Eligibility Status')
ax.set_xlabel('Credit Score')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)

plt.savefig('../reports/eda_credit_score.png', dpi=150)
plt.show()

# %% [markdown]
# ## 5. Correlation Analysis

# %%
# Select numeric columns for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in ['max_monthly_emi']]

# Add target
numeric_cols.append('max_monthly_emi')

# Compute correlation
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=16)
plt.tight_layout()
plt.savefig('../reports/eda_correlation_matrix.png', dpi=150)
plt.show()

# %% [markdown]
# ## 6. Key Insights Summary

# %%
# Generate business insights
print("\n" + "="*60)
print("BUSINESS INSIGHTS SUMMARY")
print("="*60)

print(f"\n1. Overall Eligibility Rate: {(df['emi_eligibility'] == 'Eligible').mean()*100:.1f}%")
print(f"2. High Risk Rate: {(df['emi_eligibility'] == 'High_Risk').mean()*100:.1f}%")
print(f"3. Not Eligible Rate: {(df['emi_eligibility'] == 'Not_Eligible').mean()*100:.1f}%")

# Top factors for eligibility
eligible = df[df['emi_eligibility'] == 'Eligible']
not_eligible = df[df['emi_eligibility'] == 'Not_Eligible']

print(f"\n4. Key Differentiators (Eligible vs Not Eligible):")
for col in ['credit_score', 'debt_to_income', 'affordability_ratio', 'disposable_income']:
    if col in df.columns:
        eligible_mean = eligible[col].mean()
        not_eligible_mean = not_eligible[col].mean()
        print(f"   - {col}: Eligible={eligible_mean:.2f} vs Not Eligible={not_eligible_mean:.2f}")

print(f"\n5. Best Performing EMI Scenario: {eligibility_by_scenario.idxmax()} ({eligibility_by_scenario.max():.1f}% eligibility)")
print(f"6. Worst Performing EMI Scenario: {eligibility_by_scenario.idxmin()} ({eligibility_by_scenario.min():.1f}% eligibility)")

# Recommendations
print("\n" + "="*60)
print("RECOMMENDATIONS")
print("="*60)
print("1. Focus on improving credit scores for high-risk applicants")
print("2. Vehicle and Personal Loans show lower eligibility rates - tighten criteria")
print("3. Education and Home Appliances show high eligibility - consider promotions")
print("4. Implement risk-based pricing based on credit score tiers")